In [1]:
import requests
from urllib.robotparser import RobotFileParser

BASE = "https://realpython.github.io"

r = requests.get(BASE + "/robots.txt", timeout=10)
print("robots.txt status code:", r.status_code)

rp = RobotFileParser()
rp.set_url(BASE + "/robots.txt")
rp.read()
print("Allowed to fetch /fake-jobs/ ?", rp.can_fetch("*", BASE + "/fake-jobs/"))

robots.txt status code: 404
Allowed to fetch /fake-jobs/ ? True


In [2]:
BASE2 = "https://www.arbeitnow.com"
API_URL = BASE2 + "/api/job-board-api"

HEADERS = {"User-Agent": "Student portfolio project (Elevvo internship; contact: your-github-username)"}

r2 = requests.get(BASE2 + "/robots.txt", headers=HEADERS, timeout=15)
print("robots.txt status:", r2.status_code)
print(r2.text[:600])

robots.txt status: 200
User-agent: *
Disallow:
Disallow: /*?__hstc



In [3]:
resp = requests.get(API_URL, headers=HEADERS, timeout=15)
print("status:", resp.status_code)

payload = resp.json()
print(payload.keys())

jobs = payload["data"]
print("jobs on this page:", len(jobs))
print(jobs[0].keys())

status: 200
dict_keys(['data', 'links', 'meta'])
jobs on this page: 300
dict_keys(['slug', 'company_name', 'title', 'description', 'remote', 'url', 'tags', 'job_types', 'location', 'created_at'])


In [4]:
j = jobs[0]
for k, v in j.items():
    if k != "description":
        print(k, "->", v)

print("\nDescription starts with:\n", j["description"][:400])

slug -> manager-applied-ai-engineering-codex-munich-421206
company_name -> OpenAI
title -> Manager, Applied AI Engineering (Codex)
remote -> False
url -> https://www.arbeitnow.com/jobs/companies/openai/manager-applied-ai-engineering-codex-munich-421206
tags -> ['Go To Market']
job_types -> ['Full Time']
location -> Munich, Germany
created_at -> 1793558417

Description starts with:
 <h3><strong>About the Team</strong></h3><p style="min-height:1.5em">The Applied AI Engineering team is responsible for helping customers turn frontier AI capabilities into real products, workflows, and business impact. We act as trusted technical partners across solution design, architecture, implementation, evaluation, and adoption, working alongside customers to build and scale effective AI appl


In [5]:
import os, json, time
import pandas as pd

print(payload["links"])
print(payload["meta"])

{'first': 'https://www.arbeitnow.com/api/job-board-api?page=1', 'last': None, 'prev': None, 'next': 'https://www.arbeitnow.com/api/job-board-api?page=2'}
{'current_page': 1, 'current_page_url': 'https://www.arbeitnow.com/api/job-board-api?page=1', 'from': 1, 'path': 'https://www.arbeitnow.com/api/job-board-api', 'per_page': 300, 'to': 300, 'terms': 'This is a free public API for jobs, please do not abuse. I would appreciate linking back to the site. By using the API, you agree to the terms of service present on Arbeitnow.com', 'info': 'Jobs are updated every hour and order by the `created_at` timestamp. Use `?page=` to paginate. Read more information here: https://www.arbeitnow.com/blog/job-board-api'}


In [6]:
RAW_PATH = "data/arbeitnow_raw.json"
MAX_PAGES = 5
DELAY = 2

if os.path.exists(RAW_PATH):
    with open(RAW_PATH, encoding="utf-8") as f:
        all_jobs = json.load(f)
    print("Loaded saved file:", len(all_jobs), "jobs")
else:
    all_jobs = []
    url = API_URL
    for page in range(1, MAX_PAGES + 1):
        resp = requests.get(url, headers=HEADERS, timeout=15)
        resp.raise_for_status()
        data = resp.json()
        all_jobs.extend(data["data"])
        print(f"page {page}: {len(data['data'])} jobs")

        links = data.get("links") or {}
        url = links.get("next") if isinstance(links, dict) else None
        if not url:
            break
        time.sleep(DELAY)

    with open(RAW_PATH, "w", encoding="utf-8") as f:
        json.dump(all_jobs, f, ensure_ascii=False)
    print("Saved", len(all_jobs), "jobs to", RAW_PATH)

page 1: 300 jobs
page 2: 300 jobs
page 3: 300 jobs
page 4: 300 jobs
page 5: 241 jobs
Saved 1441 jobs to data/arbeitnow_raw.json


In [7]:
df_raw = pd.DataFrame(all_jobs)
print(df_raw.shape)
print("duplicate slugs:", df_raw["slug"].duplicated().sum())

df_raw["posted_date"] = pd.to_datetime(df_raw["created_at"], unit="s")
print(df_raw["posted_date"].min(), "to", df_raw["posted_date"].max())

print(df_raw["remote"].value_counts())
print(df_raw["location"].value_counts().head(10))
print(df_raw["job_types"].astype(str).value_counts().head(5))

(1441, 10)
duplicate slugs: 0
2026-10-25 21:02:24 to 2026-11-01 20:02:20
remote
False    1343
True       98
Name: count, dtype: int64
location
Paris                          162
London                         126
Berlin                         109
Munich                          85
Amsterdam                       54
                                47
Zürich                          47
Remote job                      19
Zurich, Zürich, Switzerland     19
United Kingdom                  19
Name: count, dtype: int64
job_types
['Full Time']                                         321
[]                                                    240
['Unbefristeter arbeitsvertrag']                       97
['CDI']                                                95
['Unbefristeter arbeitsvertrag', 'berufserfahren']     94
Name: count, dtype: int64


In [8]:
from bs4 import BeautifulSoup

FAKE_URL = "https://realpython.github.io/fake-jobs/"
page = requests.get(FAKE_URL, headers=HEADERS, timeout=15)
print("status:", page.status_code)

soup = BeautifulSoup(page.content, "html.parser")
results = soup.find(id="ResultsContainer")
cards = results.find_all("div", class_="card-content")
print("job cards found:", len(cards))
print(cards[0].prettify()[:900])

status: 200
job cards found: 100
<div class="card-content">
 <div class="media">
  <div class="media-left">
   <figure class="image is-48x48">
    <img alt="Real Python Logo" src="https://files.realpython.com/media/real-python-logo-thumbnail.7f0db70c2ed2.jpg?__no_cf_polish=1"/>
   </figure>
  </div>
  <div class="media-content">
   <h2 class="title is-5">
    Senior Python Developer
   </h2>
   <h3 class="subtitle is-6 company">
    Payne, Roberts and Davis
   </h3>
  </div>
 </div>
 <div class="content">
  <p class="location">
   Stewartbury, AA
  </p>
  <p class="is-small has-text-grey">
   <time datetime="2021-04-08">
    2021-04-08
   </time>
  </p>
 </div>
 <footer class="card-footer">
  <a class="card-footer-item" href="https://www.realpython.com" target="_blank">
   Learn
  </a>
  <a class="card-footer-item" href="https://realpython.github.io/fake-jobs/jobs/senior-python-developer-0.html" target="_blank">
   Appl


In [9]:
rows = []
for card in cards:
    title = card.find("h2", class_="title")
    company = card.find("h3", class_="company")
    location = card.find("p", class_="location")
    posted = card.find("time")
    links = card.find_all("a")

    rows.append({
        "title": title.get_text(strip=True) if title else None,
        "company": company.get_text(strip=True) if company else None,
        "location": location.get_text(strip=True) if location else None,
        "posted_date": posted.get_text(strip=True) if posted else None,
        "apply_url": links[-1].get("href") if links else None,
    })

fake_df = pd.DataFrame(rows)
print(fake_df.shape)
print(fake_df.isnull().sum())
fake_df.head(3)

(100, 5)
title          0
company        0
location       0
posted_date    0
apply_url      0
dtype: int64


,title,company,location,posted_date,apply_url
0,Senior Python Developer,"Payne, Roberts and Davis","Stewartbury, AA",2021-04-08,https://realpython.github.io/fake-jobs/jobs/se...
1,Energy engineer,Vasquez-Davidson,"Christopherville, AA",2021-04-08,https://realpython.github.io/fake-jobs/jobs/en...
2,Legal executive,"Jackson, Chambers and Levy","Port Ericaburgh, AA",2021-04-08,https://realpython.github.io/fake-jobs/jobs/le...


In [10]:
fake_df["posted_date"] = pd.to_datetime(fake_df["posted_date"], errors="coerce")
print(fake_df["posted_date"].min(), "to", fake_df["posted_date"].max())
print("titles containing 'python':", fake_df["title"].str.contains("python", case=False).sum())
print(fake_df["title"].value_counts().head(5))

fake_df.to_csv("data/fake_jobs_scraped.csv", index=False)

2021-04-08 00:00:00 to 2021-04-08 00:00:00
titles containing 'python': 10
title
Python Programmer (Entry-Level)    3
Legal executive                    2
Materials engineer                 2
Software Developer (Python)        2
Futures trader                     2
Name: count, dtype: int64


In [11]:
df = df_raw.copy()
print("start:", len(df))

df = df.drop_duplicates(subset="slug")

before = len(df)
df = df.drop_duplicates(subset=["title", "company_name", "location"])
print("removed same title+company+location:", before - len(df))
print("rows now:", len(df))

print(df[["title", "company_name", "location", "description", "created_at"]].isnull().sum())
print("empty descriptions:", (df["description"].fillna("").str.strip() == "").sum())

start: 1441
removed same title+company+location: 15
rows now: 1426
title           0
company_name    0
location        0
description     0
created_at      0
dtype: int64
empty descriptions: 0


In [12]:
import re
import unicodedata
from bs4 import BeautifulSoup

COUNTRIES = {"united kingdom", "germany", "france", "netherlands", "switzerland", "austria",
             "spain", "italy", "belgium", "ireland", "poland", "sweden", "denmark", "norway",
             "finland", "portugal", "czech republic", "luxembourg", "united states"}

def strip_accents(s):
    return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()

def clean_city(loc):
    if loc is None:
        return "Unknown"
    loc = str(loc).strip()
    if loc == "" or loc.lower() in {"remote job", "remote", "anywhere", "worldwide"}:
        return "Unknown"
    first = strip_accents(loc.split(",")[0].strip())
    if first.lower() in COUNTRIES:
        return "Unknown"
    return first.title()

df["city"] = df["location"].apply(clean_city)
print(df["city"].value_counts().head(15))
print("Unknown share:", round((df["city"] == "Unknown").mean() * 100, 1), "%")

city
Paris                171
London               141
Berlin               134
Unknown              132
Munich               101
Zurich                93
Amsterdam             73
Hamburg               16
Bern                  15
Utrecht               14
Veenendaal            13
De Lier               12
Rotterdam             11
Frankfurt Am Main     11
Homeoffice            10
Name: count, dtype: int64
Unknown share: 9.3 %


In [13]:
EMAIL = re.compile(r"\S+@\S+\.\S+")
URL = re.compile(r"https?://\S+")

def clean_description(html):
    if not isinstance(html, str):
        return ""
    text = BeautifulSoup(html, "html.parser").get_text(separator=" ")
    text = EMAIL.sub(" ", text)
    text = URL.sub(" ", text)
    return re.sub(r"\s+", " ", text).strip()

df["description_clean"] = df["description"].apply(clean_description)
df["desc_words"] = df["description_clean"].str.split().str.len()

In [14]:
jobs = (df[["slug", "title", "company_name", "city", "location", "remote",
            "posted_date", "description_clean", "desc_words", "url"]]
        .rename(columns={"company_name": "company"})
        .reset_index(drop=True))

print(jobs.shape)
print(jobs["desc_words"].describe())
print(jobs["description_clean"].iloc[0][:300])

jobs.to_csv("data/arbeitnow_jobs_clean.csv", index=False)

(1426, 10)
count    1426.000000
mean      690.853436
std       325.133545
min         6.000000
25%       462.250000
50%       652.000000
75%       887.000000
max      2264.000000
Name: desc_words, dtype: float64
We are looking for brand ambassadors for our new fashion brand, interested in long-term collaboration. At HACOY we are making individuals feel comfortable with the clothing they wear, but more importantly; with themselves. Focused on home office wear, our collection presents clothing pieces that com


In [1]:
import re
import unicodedata
import pandas as pd
from bs4 import BeautifulSoup

jobs = pd.read_csv("data/arbeitnow_jobs_clean.csv")
jobs["description_clean"] = jobs["description_clean"].fillna("")

# re-apply the Cell 15 city patch (it was never saved to the file)
jobs["city"] = jobs["city"].replace({
    "Homeoffice": "Unknown", "Home Office": "Unknown",
    "Frankfurt Am Main": "Frankfurt",
})
print(jobs.shape)
print(jobs["city"].value_counts().head(8))

(1426, 10)
city
Paris        171
Unknown      142
London       141
Berlin       134
Munich       101
Zurich        93
Amsterdam     73
Hamburg       16
Name: count, dtype: int64


In [15]:
jobs["city"] = jobs["city"].replace({
    "Homeoffice": "Unknown", "Home Office": "Unknown",
    "Frankfurt Am Main": "Frankfurt",
})
print(jobs["city"].value_counts().head(40))

city
Paris                          171
Unknown                        142
London                         141
Berlin                         134
Munich                         101
Zurich                          93
Amsterdam                       73
Hamburg                         16
Frankfurt                       15
Bern                            15
Utrecht                         14
Veenendaal                      13
De Lier                         12
Rotterdam                       11
Veghel                          10
Heidelberg                       9
Stuttgart                        9
Cologne                          8
Berlin Office                    8
Manchester                       8
Paris Office                     8
Lucerne                          7
Leipzig                          6
Toulouse                         6
Paris - La Defense               6
Delft                            6
Montpellier                      6
Munchen                          5
Lausanne       

In [2]:
EN_WORDS = {"the","and","to","of","a","in","for","with","you","we","is","are","our","will",
            "be","on","as","that","this","your","or","an","at","by","from","have","it","us","who","can"}

def english_share(text):
    tokens = re.findall(r"[a-zA-Z']+", text.lower())
    if not tokens:
        return 0.0
    return sum(t in EN_WORDS for t in tokens) / len(tokens)

jobs["en_share"] = jobs["description_clean"].apply(english_share)
jobs["is_english"] = (jobs["en_share"] > 0.25) & (jobs["desc_words"] >= 50)

print("English postings:", jobs["is_english"].sum(), "of", len(jobs),
      f"({jobs['is_english'].mean() * 100:.1f}%)")

top_cities = jobs["city"].value_counts().head(8).index
print((jobs[jobs["city"].isin(top_cities)]
       .groupby("city")["is_english"].mean() * 100).round(1).sort_values())

English postings: 620 of 1426 (43.5%)
city
Paris        16.4
Hamburg      37.5
Munich       42.6
Berlin       45.5
Zurich       46.2
Unknown      63.4
Amsterdam    72.6
London       75.2
Name: is_english, dtype: float64


In [3]:
ROLE_RULES = [
    ("Data & AI",              r"data|machine learning|\bai\b|\bml\b|analytics|scientist|business intelligence|\bbi\b"),
    ("Software & IT",          r"software|developer|devops|backend|back-end|frontend|front-end|full[- ]?stack|programmer|\bit\b|cloud|security|\bsre\b"),
    ("Sales & Biz Dev",        r"sales|account executive|business development|\bsdr\b|\bbdr\b"),
    ("Marketing",              r"marketing|brand|content|seo|social media"),
    ("Finance & Accounting",   r"financ|account|audit|controller|tax"),
    ("HR & Recruiting",        r"\bhr\b|human resources|recruit|talent"),
    ("Customer Support",       r"customer|support|service desk"),
    ("Operations & Supply Chain", r"operations|supply|logistic|procurement|warehouse|purchas"),
    ("Management",             r"manager|director|head of|lead\b"),
    ("Engineering (other)",    r"engineer"),
]

def role_family(title):
    t = str(title).lower()
    for name, pattern in ROLE_RULES:
        if re.search(pattern, t):
            return name
    return "Other"

jobs["role_family"] = jobs["title"].apply(role_family)
print((jobs["role_family"].value_counts(normalize=True) * 100).round(1))

role_family
Other                        23.7
Software & IT                16.1
Management                   13.4
Data & AI                    13.2
Engineering (other)           7.2
Sales & Biz Dev               6.9
Marketing                     6.1
Finance & Accounting          4.8
Customer Support              3.4
HR & Recruiting               2.9
Operations & Supply Chain     2.2
Name: proportion, dtype: float64


In [4]:
CI = re.IGNORECASE

def rx(pattern, flags=CI):
    return re.compile(r"(?<![A-Za-z0-9])(?:" + pattern + r")(?![A-Za-z0-9])", flags)

SKILLS = {
    # programming and data
    "Python": rx(r"python"), "SQL": rx(r"sql"), "Java": rx(r"java"),
    "JavaScript": rx(r"javascript"), "TypeScript": rx(r"typescript"),
    "C++": rx(r"c\+\+"), "C#": rx(r"c#"),
    "R": rx(r"R programming|R language|RStudio", 0),
    "Excel": rx(r"(?:Microsoft|MS|Advanced) Excel|Excel spreadsheets?"),
    "Power BI": rx(r"power\s?bi"), "Tableau": rx(r"tableau"), "Looker": rx(r"looker"),
    "Snowflake": rx(r"snowflake"), "dbt": rx(r"dbt", 0), "Airflow": rx(r"airflow"),
    "Kafka": rx(r"kafka"), "Spark": rx(r"Apache Spark|PySpark"),
    "Data Analysis": rx(r"data analy(?:sis|tics)"),
    # AI
    "Machine Learning": rx(r"machine learning"), "Deep Learning": rx(r"deep learning"),
    "NLP": rx(r"nlp|natural language processing"),
    "LLM": rx(r"llms?|large language models?"),
    "Generative AI": rx(r"generative ai|genai"),
    "TensorFlow": rx(r"tensorflow"), "PyTorch": rx(r"pytorch"),
    "Scikit-learn": rx(r"scikit-?learn"),
    # engineering and cloud
    "AWS": rx(r"aws|amazon web services"), "Azure": rx(r"azure"),
    "GCP": rx(r"gcp|google cloud"), "Docker": rx(r"docker"),
    "Kubernetes": rx(r"kubernetes|k8s"), "Git": rx(r"git|github|gitlab"),
    "Linux": rx(r"linux"), "CI/CD": rx(r"ci/cd|ci-cd"),
    "REST APIs": rx(r"rest(?:ful)? apis?|restful"), "Microservices": rx(r"microservices?"),
    "React": rx(r"React(?:\.js|JS)?", 0), "Node.js": rx(r"node\.?js"),
    "Angular": rx(r"angular"), "Django": rx(r"django"),
    # business and operations
    "SAP": rx(r"SAP", 0), "ERP": rx(r"ERP", 0), "Salesforce": rx(r"salesforce"),
    "Supply Chain": rx(r"supply chain"), "Logistics": rx(r"logistics"),
    "Procurement": rx(r"procurement"), "Project Management": rx(r"project management|PMP"),
    "Agile": rx(r"agile"), "Scrum": rx(r"scrum"), "Jira": rx(r"jira"),
    # soft skills
    "Communication": rx(r"communication"), "Leadership": rx(r"leadership"),
    "Teamwork": rx(r"teamwork|team player|collaborat\w*"),
    "Problem Solving": rx(r"problem[- ]solving"),
}

eng = jobs[jobs["is_english"]].copy()
eng["text"] = eng["title"] + " " + eng["description_clean"]

for name, pattern in SKILLS.items():
    eng["skill_" + name] = eng["text"].apply(lambda t: bool(pattern.search(t)))

skill_cols = [c for c in eng.columns if c.startswith("skill_")]
print("English jobs analysed:", len(eng))
print("Jobs mentioning at least one skill:", round(eng[skill_cols].any(axis=1).mean() * 100, 1), "%")

English jobs analysed: 620
Jobs mentioning at least one skill: 94.5 %


In [5]:
freq = (eng[skill_cols].mean() * 100).round(1).sort_values(ascending=False)
freq.index = [i.replace("skill_", "") for i in freq.index]
print(freq.head(20))
print("\nNever found:", list(freq[freq == 0].index))

Teamwork              59.0
Communication         47.9
Leadership            30.8
Python                18.7
Problem Solving       15.2
AWS                    9.4
Agile                  9.2
TypeScript             9.0
LLM                    8.2
Git                    8.1
Procurement            7.7
CI/CD                  7.6
Kubernetes             7.1
Logistics              7.1
SQL                    6.8
Salesforce             6.1
React                  5.6
Project Management     4.7
Azure                  4.5
Data Analysis          4.4
dtype: float64

Never found: []
